# Hands-on Session 1: Environment Setup and Spatial Data Exploration

**Time:** 10:45-11:05  
**Lead:** Zeyuan

Goals:
- Confirm the Python/Jupyter environment.
- Load a spatial transcriptomics and spatial-CITE-seq-like dataset.
- Visualize spatial transcript and protein expression.
- Inspect simple spatial neighborhood structure.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from dgat_tutorial.data import load_tutorial_data
from dgat_tutorial.plotting import plot_spatial_feature

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_dir = repo_root / "data" / "raw"
fig_dir = repo_root / "results" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {repo_root}")

## Load Data

This notebook first looks for CSV files prepared from the official DGAT data assets in `data/raw/`. If those files are missing, it falls back to a compact synthetic dataset only so the notebook remains executable during environment checks.

In [ ]:
dataset = load_tutorial_data(data_dir, allow_demo=True)
using_official_data = all((data_dir / name).exists() for name in ["spots.csv", "transcripts.csv", "proteins.csv"])
print("Loaded official DGAT-derived tutorial data" if using_official_data else "Loaded synthetic fallback data")

spots = dataset.spots
transcripts = dataset.transcripts
proteins = dataset.proteins

spots.head()

In [ ]:
print("Spots:", spots.shape)
print("Transcript matrix:", transcripts.shape)
print("Protein matrix:", proteins.shape)

display(transcripts.iloc[:5, :5])
display(proteins.head())

## Visualize Spatial Transcript and Protein Signals

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_spatial_feature(spots, transcripts["Gene01"], "Transcript: Gene01", cmap="magma", ax=axes[0])
plot_spatial_feature(spots, proteins["CD3"], "Protein: CD3", cmap="viridis", ax=axes[1])
plt.tight_layout()
plt.savefig(fig_dir / "session01_spatial_features.png", dpi=160)
plt.show()

## Explore Spatial Neighborhoods

A common first diagnostic is to ask whether nearby spots have similar molecular profiles. Here we compute nearest neighbors from the spatial coordinates.

In [ ]:
xy = spots[["x", "y"]].to_numpy()
distance_matrix = np.sqrt(((xy[:, None, :] - xy[None, :, :]) ** 2).sum(axis=2))
neighbor_indices = np.argsort(distance_matrix, axis=1)[:, :7]
distances = np.take_along_axis(distance_matrix, neighbor_indices, axis=1)

neighbors = pd.DataFrame(
    {
        "spot_id": spots.index,
        "mean_neighbor_distance": distances[:, 1:].mean(axis=1),
        "nearest_neighbor": spots.index[neighbor_indices[:, 1]],
    }
).set_index("spot_id")

neighbors.head()

In [ ]:
ax = spots.plot.scatter(x="x", y="y", c=neighbors["mean_neighbor_distance"], cmap="cividis", figsize=(5, 4))
ax.set_title("Mean distance to 6 nearest neighbors")
ax.set_aspect("equal")
plt.tight_layout()
plt.savefig(fig_dir / "session01_neighborhood_distances.png", dpi=160)
plt.show()

## Checkpoint

Participants should now know what matrices are available, how spots are arranged spatially, and why neighborhood structure matters for spatial protein inference.